In [1]:
!pip -q install openai datasets pandas scikit-learn tqdm tenacity

In [9]:
import os
from getpass import getpass

api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

print("API key loaded securely.")

Enter your OpenAI API key: ··········
API key loaded securely.


In [10]:
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [12]:
from datasets import load_dataset

dataset = load_dataset("ChanceFocus/flare-ner")

print(dataset)
print(dataset["test"].column_names)

test_data = dataset["test"]
print("Test size:", len(test_data))
print(test_data[0])

DatasetDict({
    train: Dataset({
        features: ['query', 'answer', 'label', 'text'],
        num_rows: 408
    })
    test: Dataset({
        features: ['query', 'answer', 'label', 'text'],
        num_rows: 98
    })
    valid: Dataset({
        features: ['query', 'answer', 'label', 'text'],
        num_rows: 103
    })
})
['query', 'answer', 'label', 'text']
Test size: 98
{'query': "In the sentences extracted from financial agreements in U.S. SEC filings, identify the named entities that represent a person ('PER'), an organization ('ORG'), or a location ('LOC'). The required answer format is: 'entity name, entity type'.\nText: Subordinated Loan Agreement - Silicium de Provence SAS and Evergreen Solar Inc . 7 - December 2007 [ HERBERT SMITH LOGO ] ................................ 2007 SILICIUM DE PROVENCE SAS and EVERGREEN SOLAR , INC .\nAnswer:", 'answer': 'Silicium de Provence SAS, ORG\nEvergreen Solar Inc, ORG\nHERBERT SMITH, PER\nSILICIUM DE PROVENCE SAS, ORG\nEVERGREEN SOL

In [15]:
import os
import re
import json
import time
import pandas as pd
from tqdm import tqdm
from collections import Counter
from tenacity import retry, wait_exponential, stop_after_attempt

from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-5.5"

# Start with 5 or 10 first to test cost and format.
# Set LIMIT = None to evaluate the full test set.
LIMIT = None


NER_SCHEMA = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "entity": {
                        "type": "string",
                        "description": "The exact entity text from the input sentence."
                    },
                    "type": {
                        "type": "string",
                        "enum": ["PER", "ORG", "LOC"]
                    }
                },
                "required": ["entity", "type"],
                "additionalProperties": False
            }
        }
    },
    "required": ["entities"],
    "additionalProperties": False
}


SYSTEM_PROMPT = """
You are a financial named entity recognition model.

Task:
Extract all named entities from the input financial agreement text.

Allowed entity types:
- PER: person or role-like legal party, such as Borrower
- ORG: organization
- LOC: location

Rules:
1. Return every entity mention, including repeated mentions.
2. Use the exact entity text from the sentence.
3. Do not add explanations.
4. If there is no entity, return an empty list.
5. Return only valid JSON following the schema.
"""


@retry(wait=wait_exponential(multiplier=1, min=2, max=30), stop=stop_after_attempt(5))
def call_openai_ner(text):
    response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Text:\n{text}"
        }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "financial_ner_result",
            "schema": NER_SCHEMA,
            "strict": True
        }
    },
    reasoning={
        "effort": "low"
    },
    max_output_tokens=800
)

    return json.loads(response.output_text)


def normalize_text(s):
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s+([,.;:!?%])", r"\1", s)
    s = re.sub(r"([\(\[\{])\s+", r"\1", s)
    s = re.sub(r"\s+([\)\]\}])", r"\1", s)
    return s


def detokenize(tokens):
    s = " ".join(tokens)
    s = re.sub(r"\s+([,.;:!?%])", r"\1", s)
    s = re.sub(r"([\(\[\{])\s+", r"\1", s)
    s = re.sub(r"\s+([\)\]\}])", r"\1", s)
    return s


def bio_to_spans(labels):
    spans = []
    i = 0

    while i < len(labels):
        label = labels[i]

        if isinstance(label, str) and label.startswith("B-"):
            entity_type = label[2:]
            start = i
            i += 1

            while i < len(labels) and labels[i] == f"I-{entity_type}":
                i += 1

            end = i
            spans.append((start, end, entity_type))
        else:
            i += 1

    return spans


def predicted_entities_to_spans(text, predicted_entities, max_span_len=20):
    tokens = text.split()
    predicted_spans = []
    used_spans = set()

    for item in predicted_entities:
        entity = item.get("entity", "")
        entity_type = item.get("type", "").upper()

        if entity_type not in {"PER", "ORG", "LOC"}:
            continue

        target = normalize_text(entity)
        matched = False

        for start in range(len(tokens)):
            if matched:
                break

            for end in range(start + 1, min(len(tokens), start + max_span_len) + 1):
                if (start, end) in used_spans:
                    continue

                span_text = normalize_text(detokenize(tokens[start:end]))

                if span_text == target:
                    predicted_spans.append((start, end, entity_type))
                    used_spans.add((start, end))
                    matched = True
                    break

    return predicted_spans


def compute_metrics(gold_spans_all, pred_spans_all, gold_labels_all, pred_labels_all):
    total_tp = 0
    total_fp = 0
    total_fn = 0
    exact_match_count = 0
    total_samples = len(gold_spans_all)

    for gold_spans, pred_spans in zip(gold_spans_all, pred_spans_all):
        gold_counter = Counter(gold_spans)
        pred_counter = Counter(pred_spans)

        tp = sum((gold_counter & pred_counter).values())
        fp = sum((pred_counter - gold_counter).values())
        fn = sum((gold_counter - pred_counter).values())

        total_tp += tp
        total_fp += fp
        total_fn += fn

        if gold_counter == pred_counter:
            exact_match_count += 1

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    exact_match_accuracy = exact_match_count / total_samples if total_samples > 0 else 0.0

    correct_tokens = 0
    total_tokens = 0

    for gold_labels, pred_labels in zip(gold_labels_all, pred_labels_all):
        n = min(len(gold_labels), len(pred_labels))
        correct_tokens += sum(1 for i in range(n) if gold_labels[i] == pred_labels[i])
        total_tokens += n

    token_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "exact_match_accuracy": exact_match_accuracy,
        "token_accuracy": token_accuracy,
        "true_positive": total_tp,
        "false_positive": total_fp,
        "false_negative": total_fn
    }


def spans_to_bio_labels(spans, length):
    labels = ["O"] * length

    for start, end, entity_type in spans:
        if start < 0 or end > length or start >= end:
            continue

        labels[start] = f"B-{entity_type}"
        for i in range(start + 1, end):
            labels[i] = f"I-{entity_type}"

    return labels


rows = []
gold_spans_all = []
pred_spans_all = []
gold_labels_all = []
pred_labels_all = []

eval_data = test_data if LIMIT is None else test_data.select(range(LIMIT))

for idx, row in enumerate(tqdm(eval_data)):
    text = row["text"]
    gold_labels = row["label"]

    gold_spans = bio_to_spans(gold_labels)

    try:
        result = call_openai_ner(text)
        predicted_entities = result.get("entities", [])
    except Exception as e:
        predicted_entities = []
        print(f"Error at row {idx}: {e}")

    pred_spans = predicted_entities_to_spans(text, predicted_entities)
    pred_labels = spans_to_bio_labels(pred_spans, len(gold_labels))

    gold_spans_all.append(gold_spans)
    pred_spans_all.append(pred_spans)
    gold_labels_all.append(gold_labels)
    pred_labels_all.append(pred_labels)

    rows.append({
        "idx": idx,
        "text": text,
        "gold_answer": row["answer"],
        "predicted_entities": predicted_entities,
        "gold_spans": gold_spans,
        "pred_spans": pred_spans,
        "exact_match": Counter(gold_spans) == Counter(pred_spans)
    })

    time.sleep(0.2)


metrics = compute_metrics(
    gold_spans_all,
    pred_spans_all,
    gold_labels_all,
    pred_labels_all
)

print("\n===== GPT-5.5 Flare-NER Evaluation =====")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Precision / Correctness Rate: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"Exact Match Accuracy: {metrics['exact_match_accuracy']:.4f}")
print(f"Token Accuracy: {metrics['token_accuracy']:.4f}")
print(f"TP: {metrics['true_positive']}")
print(f"FP: {metrics['false_positive']}")
print(f"FN: {metrics['false_negative']}")

df = pd.DataFrame(rows)
df.to_csv("gpt55_flare_ner_predictions.csv", index=False)
df.head()

100%|██████████| 98/98 [04:29<00:00,  2.75s/it]


===== GPT-5.5 Flare-NER Evaluation =====
Model: gpt-5.5
Samples evaluated: 98
Precision / Correctness Rate: 0.4459
Recall: 0.7685
F1 Score: 0.5643
Exact Match Accuracy: 0.3163
Token Accuracy: 0.9335
TP: 239
FP: 297
FN: 72


,idx,text,gold_answer,predicted_entities,gold_spans,pred_spans,exact_match
0,0,Subordinated Loan Agreement - Silicium de Prov...,"Silicium de Provence SAS, ORG\nEvergreen Solar...","[{'entity': 'Silicium de Provence SAS', 'type'...","[(4, 8, ORG), (9, 12, ORG), (18, 20, PER), (24...","[(4, 8, ORG), (9, 12, ORG), (18, 20, ORG), (24...",False
1,1,SUBORDINATED LOAN AGREEMENT HERBERT SMITH LLP ...,"HERBERT SMITH, PER","[{'entity': 'HERBERT SMITH LLP', 'type': 'ORG'}]","[(3, 5, PER)]","[(3, 6, ORG)]",False
2,2,DISPUTES 10 Page 2 of 12 7 - December 2007 SUB...,"SILICIUM DE PROVENCE, ORG\nFrance, LOC\nUsine ...",[{'entity': 'SILICIUM DE PROVENCE S . A . S .'...,"[(28, 31, ORG), (49, 50, LOC), (57, 61, LOC), ...","[(49, 50, LOC), (57, 61, LOC), (59, 61, LOC), ...",False
3,3,WHEREAS : ( A ) The Borrower intends to develo...,"Borrower, PER\nFrance, LOC","[{'entity': 'Borrower', 'type': 'PER'}, {'enti...","[(6, 7, PER), (13, 14, LOC)]","[(6, 7, PER), (13, 14, LOC)]",True
4,4,( B ) Lender and Borrower have entered into an...,"Lender, PER\nBorrower, PER","[{'entity': 'Lender', 'type': 'PER'}, {'entity...","[(3, 4, PER), (5, 6, PER)]","[(3, 4, PER), (5, 6, PER)]",True
